# Dummy Experiment

Dummy notebook demonstrating how to import tools from the environment and run a simple Supervised Fine-Tuning (SFT) workflow. Intended purely for illustrative purposes.

In [ ]:
!git clone --branch feat/experiment1 --single-branch https://github.com/YassKa71/AI_agents_mini_project.git
%cd AI_agents_mini_project


Cloning into 'AI_agents_mini_project'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 66 (delta 19), reused 63 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (66/66), 921.80 KiB | 3.91 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/AI_agents_mini_project


In [ ]:
!pip install -e .

Obtaining file:///content/AI_agents_mini_project
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 143.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# IMPORTANT: You need to restart the session in colab after installing the project.

# importing needed modules
import torch
import json
import re
import ast

import pandas as pd
import numpy as np

#from research.ai_agents.environments.environment_simulation import StaticEnvironment
from research.paths import ENVIRONMENT_DATASET_PATH
from datasets import load_dataset
from trl import SFTTrainer
from trl import GRPOTrainer, GRPOConfig
from transformers import TrainingArguments
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from peft import PeftModel
from tqdm import tqdm
from google.colab import files

/content/AI_agents_mini_project/src/research/ai_agents/environment/evaluation.py:22: SyntaxWarning: invalid escape sequence '\d'
  the file: research\datasets\environment_datasets\toolmeta.json.


## Environment and Dataset

The GTA benchmark consists of 209 tool agent tasks annotated with their respective correct goal reaching sequence of thoughts and actions. For each task the agent is provided with a different set of tools and inputs and is required to answer a question or do a task in a few number of steps using exclusively the provided tools.

The reasonning process to reach the goal can be relatively complex for small language models (SLMs) as it requires gathering relevant information first using different tools and then combining the results in the correct way to reach the goal.

This benchmark provides an adequate experimental framework for our "mini-experiment" on SLM based Ai agents because it is adapted to our limited computaional resources and the environment configuration using this benchmark is easy and time efficient.

#### Clear GTA Dataset

In [ ]:
clear_gta_path = ENVIRONMENT_DATASET_PATH / "GTA_dataset" / "tasks.csv"
clear_gta_df = pd.read_csv(clear_gta_path)
clear_gta_df.head()

,Unnamed: 0,input,instruction,tools,final_answer,tool_calls
0,0,"[{'image_path': 'image/image_1.jpg', 'image_de...",How much should I pay for the beer on the tabl...,"[{""name"": ""OCR"", ""description"": ""This tool can...",[['12']],"{""CountGivenObject"": [{""arguments"": {""image"": ..."
1,1,"[{'image_path': 'image/image_3.jpg', 'image_de...",I want to buy a PS5 for each child in the phot...,"[{""name"": ""Calculator"", ""description"": ""A calc...",[['1919.96']],"{""CountGivenObject"": [{""arguments"": {""image"": ..."
2,2,"[{'image_path': 'image/image_7.jpg', 'image_de...",How many cups of water do the people in the p...,"[{""name"": ""Calculator"", ""description"": ""A calc...",[['27']],"{""OCR"": [{""arguments"": {""image"": ""image/image_..."
3,3,"[{'image_path': 'image/image_9.jpg', 'image_de...",I need to prepare twelve servings of this dis...,"[{""name"": ""OCR"", ""description"": ""This tool can...",[['2']],"{""OCR"": [{""arguments"": {""image"": ""image/image_..."
4,4,"[{'image_path': 'image/image_11.jpg', 'image_d...",I want to buy a dog toy for each dog in the p...,"[{""name"": ""Calculator"", ""description"": ""A calc...",[['79.96']],"{""OCR"": [{""arguments"": {""image"": ""image/image_..."


#### Finetuning Dataset

In [ ]:
# Load dataset
finetuning_dataset_path = ENVIRONMENT_DATASET_PATH / "GTA_dataset" / "finetuning_dataset.jsonl"
dataset = load_dataset("json", data_files={"train": str(finetuning_dataset_path)})
ds = dataset["train"]
dataset_df = ds.select(range(5)).to_pandas()
print("Dataset overview:")
dataset_df.head()

Generating train split: 0 examples [00:00, ? examples/s]

Dataset overview:


,prompt,completion,task_id
0,"[{'role': 'system', 'content': 'You are an exp...","[{'role': 'assistant', 'content': 'Thought: Th...",0
1,"[{'role': 'system', 'content': 'You are an exp...","[{'role': 'assistant', 'content': 'Thought: No...",0
2,"[{'role': 'system', 'content': 'You are an exp...","[{'role': 'assistant', 'content': 'Thought: No...",0
3,"[{'role': 'system', 'content': 'You are an exp...","[{'role': 'assistant', 'content': 'Action: {'n...",0
4,"[{'role': 'system', 'content': 'You are an exp...","[{'role': 'assistant', 'content': 'Thought: To...",1


In [ ]:
GROUP_COL = "task_id"
SEED = 42

# split by unique group ids, then select rows by indices
def split_by_group(ds, group_col, test_size, seed):
    # Get all group ids
    group_values = ds[group_col]
    unique_groups = np.array(sorted(set(group_values)))

    rng = np.random.default_rng(seed)
    rng.shuffle(unique_groups)

    n_test_groups = int(round(len(unique_groups) * test_size))
    test_groups = set(unique_groups[:n_test_groups])
    train_groups = set(unique_groups[n_test_groups:])

    # Build row indices for each split
    train_idx = [i for i, g in enumerate(group_values) if g in train_groups]
    test_idx  = [i for i, g in enumerate(group_values) if g in test_groups]

    return ds.select(train_idx), ds.select(test_idx)


# Split into train/test with disjoint task_ids
train_dataset, test_dataset = split_by_group(
    ds, GROUP_COL, test_size=0.4, seed=SEED
)

# Split train into SFT vs GRPO with disjoint task_ids
sft_train_dataset, grpo_train_dataset = split_by_group(
    train_dataset, GROUP_COL, test_size=0.5, seed=SEED
)

# Display lengths
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"SFT train dataset size: {len(sft_train_dataset)}")
print(f"GRPO train dataset size: {len(grpo_train_dataset)}")

# Display task ids for each split
sft_group_values = sft_train_dataset[GROUP_COL]
sft_tasks = np.array(sorted(set(sft_group_values)))
print(f"\n SFT task ids: \n{sft_tasks}")

grpo_group_values = grpo_train_dataset[GROUP_COL]
grpo_tasks = np.array(sorted(set(grpo_group_values)))
print(f"\n GRPO task ids: \n{grpo_tasks}")

train_group_values = train_dataset[GROUP_COL]
train_tasks = np.array(sorted(set(train_group_values)))
print(f"\n All train task ids: \n{train_tasks}")

test_group_values = test_dataset[GROUP_COL]
test_tasks = np.array(sorted(set(test_group_values)))
print(f"\n All test task ids: \n{test_tasks}")

Train dataset size: 406
Test dataset size: 266
SFT train dataset size: 203
GRPO train dataset size: 203

 SFT task ids: 
['1' '11' '112' '114' '118' '119' '125' '126' '13' '134' '135' '141' '142'
 '146' '15' '151' '152' '153' '156' '157' '160' '162' '164' '165' '169'
 '172' '173' '180' '181' '182' '188' '190' '193' '196' '197' '198' '199'
 '202' '205' '212' '216' '219' '221' '224' '225' '227' '27' '33' '37' '38'
 '4' '40' '46' '54' '55' '56' '58' '63' '64' '7' '77' '78' '8' '80' '86'
 '90' '91' '94' '98']

 GRPO task ids: 
['0' '102' '103' '107' '108' '110' '111' '113' '115' '117' '129' '130'
 '133' '136' '137' '138' '139' '140' '150' '154' '155' '167' '168' '175'
 '177' '18' '186' '191' '20' '200' '203' '206' '208' '209' '210' '214'
 '22' '220' '222' '223' '226' '23' '24' '25' '3' '30' '34' '36' '42' '43'
 '50' '51' '52' '57' '59' '66' '67' '68' '69' '70' '71' '72' '74' '82'
 '83' '84' '85' '97']

 All train task ids: 
['0' '1' '102' '103' '107' '108' '11' '110' '111' '112' '113' '114

## Supervized finetuning on 50% of training dataset

In [ ]:
def load_base(model_name):
    # 4-bit quantization config (QLoRA)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    )
    base = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    )
    base.config.use_cache = False
    base = prepare_model_for_kbit_training(base)
    return base

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# load quantized model
model = load_base(model_name)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# Attach LoRA adapter
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Training arguments
training_args = TrainingArguments(
    output_dir="./out_sft",
    per_device_train_batch_size=3,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=5,
    logging_steps=10,
    save_strategy="no",
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    fp16=not torch.cuda.is_available(),
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    report_to="none",
)

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=sft_train_dataset,
)
trainer.train()
trainer.save_model("./out_sft/final")

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:2111: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.061100
20,0.756400
30,0.583100
40,0.566000


In [ ]:
!zip -r ./out_sft.zip ./out_sft
files.download('./out_sft.zip')

  adding: out_sft/ (stored 0%)
  adding: out_sft/final/ (stored 0%)
  adding: out_sft/final/README.md (deflated 65%)
  adding: out_sft/final/merges.txt (deflated 57%)
  adding: out_sft/final/special_tokens_map.json (deflated 69%)
  adding: out_sft/final/vocab.json (deflated 61%)
  adding: out_sft/final/added_tokens.json (deflated 67%)
  adding: out_sft/final/adapter_config.json (deflated 58%)
  adding: out_sft/final/tokenizer.json (deflated 81%)
  adding: out_sft/final/chat_template.jinja (deflated 71%)
  adding: out_sft/final/adapter_model.safetensors (deflated 22%)
  adding: out_sft/final/training_args.bin (deflated 52%)
  adding: out_sft/final/tokenizer_config.json (deflated 89%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>